### ROW REMOVAL

In [5]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score
import os

import import_ipynb
import importlib
import functions as fc
importlib.reload(fc)

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


/Users/elif/Desktop/Projects/privacy_in_ml


In [6]:
diabetes_train = pd.read_csv("datasets/diabetes_train.csv")
diabetes_test = pd.read_csv("datasets/diabetes_test.csv")

X_train = diabetes_train.drop('Outcome', axis=1)
X_test = diabetes_test.drop('Outcome', axis=1)
y_train = diabetes_train['Outcome']
y_test = diabetes_test['Outcome']


In [7]:
def preprocess(X_tr, X_te):
    scaler = StandardScaler()
    return scaler.fit_transform(X_tr), scaler.transform(X_te)


In [8]:
n_trials = 50
np.random.seed(42)

output_file = "results/row_removal_diabetes.xlsx"

#Baseline
X_train_sc, X_test_sc = preprocess(X_train, X_test)

model = LogisticRegression(C=1, max_iter=1000, random_state=42)
model.fit(X_train_sc, y_train)

print("Training set (full data): ")
fc.predict_binary(model, X_train_sc, y_train, conf_matrix=False)

fc.predict_binary_save_results(
    model,
    X_test_sc,
    y_test,
    conf_matrix=False,
    perturbation_type="RowRemoval",
    epsilon=0,
    row_id=None,
    output_file=output_file
)

#Trials
for trial in range(1, n_trials + 1):
    idx = np.random.randint(0, X_train.shape[0])
    removed_label = X_train.index[idx]

    X_train_reduced = X_train.drop(index=removed_label)
    y_train_reduced = y_train.drop(index=removed_label)

    X_train_reduced_sc, X_test_reduced_sc = preprocess(X_train_reduced, X_test)

    trial_model = LogisticRegression(C=1, max_iter=1000, random_state=42)
    trial_model.fit(X_train_reduced_sc, y_train_reduced)

    print(f"Trial {trial}/{n_trials} - removed row {removed_label}")
    print("Training set: ")
    fc.predict_binary(trial_model, X_train_reduced_sc, y_train_reduced, conf_matrix=False)

    fc.predict_binary_save_results(
        trial_model,
        X_test_reduced_sc,
        y_test,
        conf_matrix=False,
        perturbation_type="RowRemoval",
        epsilon=trial,
        row_id=removed_label,
        output_file=output_file
    )


Training set (full data): 
---------------------------------------
Accuracy: 0.7801
Precision: 0.7241
Recall: 0.5915
F1 Score: 0.6512
---------------------------------------
---------------------------------------
Accuracy: 0.7662
Precision: 0.6863
Recall: 0.6364
F1 Score: 0.6604
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Trial 1/50 - removed row 102
Training set: 
---------------------------------------
Accuracy: 0.7798
Precision: 0.7241
Recall: 0.5915
F1 Score: 0.6512
---------------------------------------
---------------------------------------
Accuracy: 0.7662
Precision: 0.6863
Recall: 0.6364
F1 Score: 0.6604
---------------------------------------
Saved results to results/row_removal_diabetes.xlsx
Trial 2/50 - removed row 435
Training set: 
---------------------------------------
Accuracy: 0.7814
Precision: 0.7241
Recall: 0.5943
F1 Score: 0.6528
---------------------------------------
---------------------------------------
Accuracy